In [3]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [2]:
print('length of dataset (chars): ', len(text))

length of dataset (chars):  1115394


In [3]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('chars=' + ''.join(chars))
print(f'vocab_size={vocab_size}')

chars=
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size=65


In [5]:
stoi = { ch:i for i, ch in enumerate(chars) }
itos = { stoi[ch]:ch for ch in stoi }

# We need to encode and decode the numbers because neural networks only understand numbers
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hello world'))
print(decode(encode('hello world')))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [6]:
import torch

# Encode the text dataset and store it into a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [7]:
# Split up the data into train and validation sets
# First 90% for training, remaining 10% for validation
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f'when input is {context} target is {target}')

when input is tensor([18]) target is 47
when input is tensor([18, 47]) target is 56
when input is tensor([18, 47, 56]) target is 57
when input is tensor([18, 47, 56, 57]) target is 58
when input is tensor([18, 47, 56, 57, 58]) target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) target is 58


In [9]:
torch.manual_seed(1337)
batch_size = 4 # number of independent sequences will be processed before updating gradients
block_size = 8 # max context length (in this case, number of characters to form a context)

def get_batch(split):
    data = train_data if split == 'train' else val_data

    # Generate random offsets in the data
    offsets = torch.randint(len(data) - block_size, size=(batch_size,))

    # Inputs
    x = torch.stack([data[i:i+block_size] for i in offsets])

    # Targets
    y = torch.stack([data[i+1:i+1+block_size] for i in offsets])

    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('------------')

for batch in range(batch_size):
    for i in range(block_size):
        context = xb[batch, :i+1]
        target = yb[batch, i]
        print(f'Input: {context.tolist()}, Target: {target}')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
------------
Input: [24], Target: 43
Input: [24, 43], Target: 58
Input: [24, 43, 58], Target: 5
Input: [24, 43, 58, 5], Target: 57
Input: [24, 43, 58, 5, 57], Target: 1
Input: [24, 43, 58, 5, 57, 1], Target: 46
Input: [24, 43, 58, 5, 57, 1, 46], Target: 43
Input: [24, 43, 58, 5, 57, 1, 46, 43], Target: 39
Input: [44], Target: 53
Input: [44, 53], Target: 56
Input: [44, 53, 56], Target: 1
Input: [44, 53, 56, 1], Target: 58
Input: [44, 53, 56, 1, 58], Target: 46
Input: [44, 53, 56, 1, 58, 46], Target: 39
Input: [44, 53, 56, 1, 58, 46, 39], Target: 58
Input: [44, 53, 56, 1, 58, 46, 39, 58], Target: 1
Inp

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Each token directly reads off the logits for the next token from a lookup table
        # Maps a token (vocab_size possible tokens) to a vector of size vocab_size
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, indices, targets=None):
        # `idx` is the inputs, `targets` is the expected output
        # They are both (B,T) tensor of integers
        # Batch, time steps/tokens, channels
        # Channels means: for each batch, for each token, we have a vector of C = 65 channels
        logits = self.token_embedding_table(indices) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            # Cross-entropy conceptually wants:
            # logits: (# of examples, # of classes)
            # targets: (# of examples)
            # cross_entropy here also automatically does argmax on the row in logits
            # and uses that value to cross-entropy with the element in targets
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, indices, max_new_tokens):
        # Indices is (B,T) tensor of indices in the current context
        for _ in range(max_new_tokens):
            # Receive the predictions. Since targets=None, the reshaping does not happen for logits
            logits, loss = self(indices) # logits: (B,T,C) 

            # Logits for only the last time-step to generate the next character
            # This kind of defeats the idea of "context" because it is solely using the last char
            # to predict the next one, but this is the bigram model. And as you can guess, it sucks
            logits = logits[:, -1, :] # becomes (B,C) for T=-1

            # Apply softmax to get probabilities
            probabilities = F.softmax(logits, dim=-1)

            # Sample from the distribution
            next_idx = torch.multinomial(probabilities, num_samples=1) # (B,1)

            indices = torch.cat((indices, next_idx), dim=1) # (B,T+1)

        return indices

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

indices = torch.zeros((1,1), dtype=torch.long)
print(decode(model.generate(indices, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [11]:
# Create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [12]:
batch_size = 32
for steps in range(5000):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.536130905151367


In [13]:
print(decode(model.generate(indices=torch.zeros((1,1), dtype=torch.long), max_new_tokens=1000)[0].tolist()))





CHNCKIViver HelozR'd jemiok ft hat fo is -mZARSure, Yje'd ureckha;
ENCEngiAs! smiTId
W:
CE:Pich toto ito,'r. alyy f?
E-b, zoll terat;&suck:
ThgCA:
POxcotDUCENC:
Ramitr,
CHo IEir -kzowakow Chearouy ino in usate't we cksw,
JzPY:
Sof m Vbs, hatarakis,bereFotomampure,,
W:CIN wlflin: ay ced isordwhau'TI w!AUCUNSome! b!
nfry andilk!an!
DITh
If iloinoth hithcot; e zCAr,
june, thes aithak;E:

Sen ing ve ce athly wnd hrt ve teogs se.
VOUMpbe havefulpimngUFLUGott and:
ARIUSa-PHEENV
PE:
Ap arotegnYBupre found ds de r s; gq-rilarckememeve hu tDUxfonsopouQUCl:&PHai
Agr tylo o'

Nof3ullicdf Thiscolly s3Q! ty iscad ort,SBEy,
Alathal thid?-t Rofu,
Aclldato ld se ne, hathJXXMAm; lr ay menienearsenx'BUELLIO Bemewiadl and lit wothorou t
R
Ad ther cowaf stot?3Wh$'s thim

Yo'stTMqR:?spp:
ARo GruD GLYeirt izr EHUSusoS:
Ll

Mat gKI'sthe, hok.
Fofelinsth,

'lolly!
To, q; s SDWhagapxErers agq-lullulea;
Ame w ANYRI: totof Tr n,

Y:
xccende gssthoiun norode,
A se iJGhe: rre sammoun 
TINXmsHot'Thes smeth$R:
P

In [ ]:
# Even though there are a good number of parameters
# Since this is straight input -> output with no layers in between, it sucks
print(sum(p.numel() for p in model.parameters() if p.requires_grad))

4225


# Mathematical trick in self-attention

In [14]:
# Example

torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [49]:
# We want x[b,t] = mean_{i<=t} x[b,i]
# BOW = Bag of words is a term used when you are just averaging up things
x_bow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        x_prev = x[b,:t+1] # (t,C). Also note that x_prev includes the current vector too
        x_bow[b, t] = torch.mean(x_prev, 0) # (C,)

In [16]:
# First batch
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [23]:
x_bow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [ ]:
# Example demonstrating how matrix multiplication can be used for "weighted aggregation"
# Weight aggregation: combining several values together, but giving some values more importance
#                     than others--hence the "weight" part
# This basically does the above process of calculating the average logit value across all previous
# tokens but in one matrix multiplication instead of two for loopsk

torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))

# Normalize each row via broadcasting
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b

print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [85]:
# Version 2
weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True)
x_bow2 = weights @ x # (B, T, T) @ (B, T, C) -> (B, T, C)
print(x_bow2)
print(torch.allclose(x_bow, x_bow2, atol=1e-4, rtol=1e-4))

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])
True


In [ ]:
# Version 3 (softmax)
tril = torch.tril(torch.ones(T, T))
weights = torch.zeros((T,T))
weights = weights.masked_fill(tril == 0, float('-inf')) # And softmax(-inf) = 0
weights = F.softmax(weights, dim=-1)
x_bow3 = weights @ x # (T,T) @ (B,T,C) -> (B,T,C)
print(x_bow3)
print(torch.allclose(x_bow, x_bow3, atol=1e-4, rtol=1e-4))

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])
True
torch.Size([8, 8])


In [105]:
# Version 4 (self-attention)
torch.manual_seed(1337)

# Batch, time, channels
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# Single head that performs self-attention
head_size = 16
query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
q = query(x) #(B,T,head_size)
k = key(x) # (B,T,head_size)
weights = q @ k.transpose(-2, -1) # (B, T, T)

tril = torch.tril(torch.ones(T, T))
# No need for this below:
# weights = torch.zeros((T, T))
weights = weights.masked_fill(tril == 0, float('-inf'))
weights = F.softmax(weights, dim=-1)
out = weights @ x

out.shape

torch.Size([4, 8, 32])

In [106]:
weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5877, 0.4123, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4457, 0.2810, 0.2733, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2220, 0.7496, 0.0175, 0.0109, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0379, 0.0124, 0.0412, 0.0630, 0.8454, 0.0000, 0.0000, 0.0000],
         [0.5497, 0.2187, 0.0185, 0.0239, 0.1831, 0.0062, 0.0000, 0.0000],
         [0.2576, 0.0830, 0.0946, 0.0241, 0.1273, 0.3627, 0.0507, 0.0000],
         [0.0499, 0.1052, 0.0302, 0.0281, 0.1980, 0.2657, 0.1755, 0.1474]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4289, 0.5711, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5413, 0.1423, 0.3165, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0635, 0.8138, 0.0557, 0.0669, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4958, 0.0758, 0.2224, 0.0156, 0.1905, 0.0000, 0.0000, 0.0000],
         [0.3957, 0.112

In [3]:
import torch
from torch import nn

# Batches, Context Size (times), Embedding Size (channels)
batches = 32
context_size = 8
embedding_size = 128

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(embedding_size, head_size, bias=False)
        self.query = nn.Linear(embedding_size, head_size, bias=False)
        self.value = nn.Linear(embedding_size, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))

    def forward(self, x):
        K = self.key(x)
        Q = self.query(x)

        print("x:", x.shape)
        print("K:", K.shape)
        print("Q:", Q.shape)


# Create the input
x = torch.randn(batches, context_size, embedding_size)

# Create a head
head = Head(head_size=64)

# Run x through the head
head(x)

x: torch.Size([32, 8, 128])
K: torch.Size([32, 8, 64])
Q: torch.Size([32, 8, 64])


In [6]:
from torch.nn import functional as F

B, T, C = 32, 8, 128
head_size = 16

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embedding_size, head_size, bias=False)
        self.query = nn.Linear(embedding_size, head_size, bias=False)
        self.value = nn.Linear(embedding_size, head_size, bias=False)

        # Used for masking (dimesion: context size by context size)
        self.register_buffer('tril', torch.tril(torch.ones(T, T)))

    def forward(self, x):
        B, T, C = x.shape
        K = self.key(x) # (B, T, d_k)
        Q = self.query(x) # (B, T, d_k)

        # Calculate softmax(Q * K^T/sqrt(d_k))
        weights = Q @ K.transpose(-2, -1) * C ** -0.5
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weights = F.softmax(weights, dim=-1) # (B, T, T)

        V = self.value(x) # (B, T, d_k)
        out = weights @ V # (B, T, d_k)

        return out

x = torch.randn(B, T, C)
head = Head(head_size)
out = head(x)
print(out.shape)

torch.Size([32, 8, 16])


In [3]:
import torch
device = 'cuda'
context = torch.zeros((1, 1), dtype=torch.long, device=device)
context

tensor([[0]], device='cuda:0')